In [1]:
# CPU-сборки. НЕ используем cu121 — на этой машине нет NVIDIA GPU.
!pip install -q gdown datasets langchain-core langchain-community \
    langchain-huggingface langchain-ollama langchain-text-splitters \
    faiss-cpu sentence-transformers gradio
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu


[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from datasets import load_dataset
from langchain_core.documents import Document
import numpy as np

ds = load_dataset("TryDotAtwo/russian-legal-ner")

all_docs = []
for split in ds:
    for i, row in enumerate(ds[split]):
        all_docs.append(Document(
            page_content=row["text"],
            metadata={"split": split, "doc_id": f"{split}_{i}"},
        ))

print(f"Загружено документов: {len(all_docs)}")

lengths = [len(d.page_content) for d in all_docs]
print(f"Длины: min={min(lengths)}, median={int(np.median(lengths))}, max={max(lengths)}")

C:\Users\Vitaliy\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Загружено документов: 97668
Длины: min=755, median=4473, max=72357


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 700 симв. — компромисс: меньше контекст → быстрее генерация на CPU.
# 150 overlap (~21%) — сохраняет смысл на стыках.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=150,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(all_docs)
print(f"Получено чанков: {len(chunks)}")

Получено чанков: 1194508


In [4]:
import os, torch
from langchain_huggingface import HuggingFaceEmbeddings

# Ограничиваем число потоков torch — иначе на 4-ядерном CPU
# потоки конкурируют и всё становится медленнее.
torch.set_num_threads(6)          # 6 из 8 логических
os.environ["OMP_NUM_THREADS"] = "6"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small",   # 118M, в 3–4× быстрее base на CPU
    model_kwargs={"device": "cpu"},
    encode_kwargs={
        "batch_size": 16,                 # под 4 физических ядра
        "normalize_embeddings": True,
    },
)
print("Embeddings готовы (CPU, e5-small)")

C:\Users\Vitaliy\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vitaliy\.cache\huggingface\hub\models--intfloat--multilingual-e5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\Vitaliy\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cac

Embeddings готовы (CPU, e5-small)


In [5]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

# e5 требует префикс "passage: "
chunks_e5 = [
    Document(page_content=f"passage: {c.page_content}", metadata=c.metadata)
    for c in chunks
]
print(f"Чанков для индексации: {len(chunks_e5)}")

BATCH = 64   # маленький батч — меньше пик RAM, стабильнее на CPU

vectorstore = None
for i in range(0, len(chunks_e5), BATCH):
    batch = chunks_e5[i:i + BATCH]
    if vectorstore is None:
        vectorstore = FAISS.from_documents(batch, embeddings)
    else:
        vectorstore.add_documents(batch)
    if (i // BATCH) % 5 == 0:
        print(f"Проиндексировано: {i + len(batch)}/{len(chunks_e5)}")

vectorstore.save_local("faiss_index")
print(f"Индекс сохранён. Векторов: {vectorstore.index.ntotal}")

C:\Users\Vitaliy\AppData\Local\Temp\ipykernel_692\2252310996.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Чанков для индексации: 1194508
Проиндексировано: 64/1194508
Проиндексировано: 384/1194508
Проиндексировано: 704/1194508
Проиндексировано: 1024/1194508
Проиндексировано: 1344/1194508
Проиндексировано: 1664/1194508
Проиндексировано: 1984/1194508
Проиндексировано: 2304/1194508
Проиндексировано: 2624/1194508
Проиндексировано: 2944/1194508
Проиндексировано: 3264/1194508
Проиндексировано: 3584/1194508
Проиндексировано: 3904/1194508
Проиндексировано: 4224/1194508
Проиндексировано: 4544/1194508
Проиндексировано: 4864/1194508
Проиндексировано: 5184/1194508
Проиндексировано: 5504/1194508
Проиндексировано: 5824/1194508
Проиндексировано: 6144/1194508
Проиндексировано: 6464/1194508
Проиндексировано: 6784/1194508
Проиндексировано: 7104/1194508
Проиндексировано: 7424/1194508
Проиндексировано: 7744/1194508
Проиндексировано: 8064/1194508
Проиндексировано: 8384/1194508
Проиндексировано: 8704/1194508
Проиндексировано: 9024/1194508
Проиндексировано: 9344/1194508
Проиндексировано: 9664/1194508
Проиндексиро

In [6]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import ChatOllama

loaded_store = FAISS.load_local(
    "faiss_index", embeddings,
    allow_dangerous_deserialization=True,
)

def retrieve(query, k=2):
    """k=2 по умолчанию — меньше контекста → быстрее генерация на CPU."""
    docs = loaded_store.similarity_search(f"query: {query}", k=k)
    context = "\n\n---\n\n".join(
        f"[{d.metadata.get('doc_id', '?')}]\n{d.page_content.replace('passage: ', '', 1)}"
        for d in docs
    )
    sources = [d.metadata.get("doc_id", "?") for d in docs]
    return context, sources

def build_prompt(question, context):
    return f"""Ты корпоративный ассистент. Отвечай ТОЛЬКО на основе контекста ниже.
Если ответа в контексте нет — ответь ровно: "В предоставленных документах ответа нет".
Если твои общие знания противоречат контексту — приоритет у контекста.
Не выдумывай факты, штрафы, даты. Отвечай кратко.

Контекст:
{context}

Вопрос: {question}
Ответ:"""

# Mistral через Ollama на CPU.
# num_ctx=4096 — чтобы 2 чанка по 700 симв. точно влезли, но не раздувать KV-кэш.
# num_thread=6 — оставляем 2 потока системе (UI, Gradio).
llm = ChatOllama(
    model="mistral",
    num_ctx=4096,
    num_thread=6,
    temperature=0.1,
    base_url="http://localhost:11434",
)

def rag_pipeline(query: str, k: int = 2):
    context, sources = retrieve(query, k=k)
    prompt = build_prompt(query, context)
    response = llm.invoke(prompt)
    return response.content, sources

In [7]:
print("=" * 70)
print("Тест 1 (по базе):")
ans, src = rag_pipeline("Какие документы подтверждают вину Кондратьева С.А.?")
print(ans); print("Источники:", src)

print("=" * 70)
print("Тест 2 (вне базы):")
ans, src = rag_pipeline("Как испечь шарлотку с яблоками?")
print(ans); print("Источники:", src)

print("=" * 70)
print("Тест 3 (конфликт):")
ans, src = rag_pipeline("Какой штраф за нарушение правил дорожного движения в 2024 году?")
print(ans); print("Источники:", src)

Тест 1 (по базе):
 Протокол об административном правонарушении №52 МА 575096 от <ДАТА6> года, акт освидетельствования на состояние опьянения 52 СС 089933, протокол об отстранении от управления транспортным средством 52 ВУ 376397.
Источники: ['train_62774', 'train_30952']
Тест 2 (вне базы):
 В предоставленных документах ответа нет.
Источники: ['train_52551', 'train_54045']
Тест 3 (конфликт):
 В предоставленных документах не указано конкретного года. Однако, в соответствии с п. 3 ст. 12.16 КоАП РФ, наложение административного штрафа в размере пяти тысяч рублей или лишение права управления транспортными средствами на срок от четырех до шести месяцев может быть применено к нарушению правил дорожного движения во встречном направлении по дороге с односторонним движением в любой год, в том числе и в 2024 году.
Источники: ['train_7047', 'train_49174']


In [8]:
import time

QUERY = "Какие документы подтверждают вину Кондратьева С.А.?"
print(f"{'k':>3} | {'время, с':>10} | {'симв. ответа':>12}")
print("-" * 34)
for k in (1, 2, 3):
    t0 = time.time()
    ans, src = rag_pipeline(QUERY, k=k)
    dt = time.time() - t0
    print(f"{k:>3} | {dt:>10.1f} | {len(ans):>12}")

  k |   время, с | симв. ответа
----------------------------------
  1 |       16.3 |          195
  2 |       22.3 |          211
  3 |       22.3 |          211


In [9]:
import gradio as gr

def chat_fn(message, history):
    answer, sources = rag_pipeline(message, k=2)
    src = ", ".join(str(s) for s in sources) if sources else "—"
    return f"{answer}\n\n_Источники: {src}_\n\n_⏱ CPU-генерация может занять 30–90 с._"

demo = gr.ChatInterface(
    chat_fn,
    title="Корпоративный RAG-ассистент (CPU)",
    description=(
        "Mistral 7B (Ollama) + FAISS + intfloat/multilingual-e5-small. "
        "Работает полностью локально, без внешних API."
    ),
    examples=[
        "Какие документы подтверждают вину Кондратьева С.А.?",
        "Как испечь шарлотку с яблоками?",
        "Какой штраф за нарушение правил дорожного движения в 2024 году?",
    ],
)
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [10]:
requirements = """datasets
langchain-core
langchain-community
langchain-huggingface
langchain-ollama
langchain-text-splitters
faiss-cpu
sentence-transformers
gradio
torch
torchvision
torchaudio
"""
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements)
print("requirements.txt создан")

requirements.txt создан
